In [2]:
%pip install pandas numpy openpyxl scipy matplotlib

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 4.4 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 5.0 MB/s eta 0:00:0000:0100:01
  Using cached openpyxl-3.1.5-py2.py3-none-any.whl (250 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.3/20.3 MB 4.8 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.1/8.1 MB 5.0 MB/s eta 0:00:0000:0100:01
  Using cached et_xmlfile-2.0.0-py3-none-any.whl (18 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 270.1/270.1 kB 5.4 MB/s eta 0:00:00a 0:00:01
  Using cached cycler-0.12.1-py3-none-any.whl (8.3 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 5.2 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.3/65.3 kB 4.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 4.8 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.8/122.8 kB 3.7 MB/s eta 0:00:00

[notice] A new release of pip available: 22.3 

In [3]:
import pandas as pd
import numpy as np

print("Starting data load and cleaning...")

# --------------------------------------------------------
# 1. LOAD STATIC DATA & FILTER REGION
# --------------------------------------------------------
# Load the static dataset
static_df = pd.read_excel('Data_2026/Static_2025.xlsx')

# Filter strictly for Emerging Markets (EM)
em_firms = static_df[static_df['Region'] == 'EM'].copy()

# Drop rows where ISIN is entirely missing
em_firms.dropna(subset=['ISIN'], inplace=True)
valid_em_isins = em_firms['ISIN'].unique().tolist()

print(f"-> Found {len(valid_em_isins)} valid EM firms in static data.")

# --------------------------------------------------------
# 2. LOAD & CLEAN MONTHLY RETURN INDEX (RI)
# --------------------------------------------------------
# Load the monthly Return Index data (index_col=0 sets dates as the index)
ri_monthly = pd.read_excel('Data_2026/DS_RI_T_USD_M_2025.xlsx', index_col=0)

# Filter columns to only include our valid EM firms
# (Intersection of EM ISINs and the columns actually present in the RI dataset)
em_columns_in_ri = [isin for isin in valid_em_isins if isin in ri_monthly.columns]
ri_em = ri_monthly[em_columns_in_ri].copy()

# Project Rule: Treat any RI price below 0.5 as a missing value (NaN)
ri_em[ri_em < 0.5] = np.nan

# --------------------------------------------------------
# 3. CALCULATE SIMPLE RETURNS
# --------------------------------------------------------
# Compute simple monthly returns (price_t - price_t-1) / price_t-1
# The pct_change() function does this automatically
returns_em = ri_em.pct_change()

print(f"-> Cleaned RI data and calculated simple returns for {len(returns_em.columns)} EM firms.")

# --------------------------------------------------------
# 4. LOAD SCOPE 1 & 2 EMISSIONS DATA
# --------------------------------------------------------
# Load the yearly carbon data since your group strategy is Scope 1+2
scope1_df = pd.read_excel('Data_2026/DS_CO2_SCOPE_1_Y_2025.xlsx', index_col=0)
scope2_df = pd.read_excel('Data_2026/DS_CO2_SCOPE_2_Y_2025.xlsx', index_col=0)

# Filter carbon datasets to only include our EM firms
scope1_em = scope1_df[[isin for isin in valid_em_isins if isin in scope1_df.columns]].copy()
scope2_em = scope2_df[[isin for isin in valid_em_isins if isin in scope2_df.columns]].copy()

print("-> Scope 1 and Scope 2 data loaded and filtered.")
print("Data preparation complete!")

Starting data load and cleaning...
-> Found 702 valid EM firms in static data.
-> Cleaned RI data and calculated simple returns for 0 EM firms.
-> Scope 1 and Scope 2 data loaded and filtered.
Data preparation complete!
